# Tetris Afterstate DQN — Pure TensorFlow
No tf_agents. Uses afterstate design: simulate every valid placement, extract
board features, pick the one the value network scores highest.

**Architecture:** V(s') → scalar  (board quality after placement)
**Training:**     TD(0) with soft target-network updates
**Logging:**      TensorBoard + CSV (mirrors old notebook structure)


In [ ]:
# pip install tensorboard

In [ ]:
import os, csv, random, datetime
from pathlib import Path
from collections import deque

import numpy as np
import tensorflow as tf
from tensorflow import keras

from tetris import Tetris

print("TF:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

In [ ]:
# Feature extraction
def _col_heights(board: np.ndarray) -> np.ndarray:
    rows, cols = board.shape
    heights = np.zeros(cols, dtype=np.float32)
    for col in range(cols):
        for row in range(rows):
            if board[row, col] == 1:
                heights[col] = rows - row
                break
    return heights

def _count_holes(board: np.ndarray) -> int:
    holes = 0
    for col in range(board.shape[1]):
        found = False
        for row in range(board.shape[0]):
            if board[row, col] == 1: found = True
            elif found: holes += 1
    return holes

def _bumpiness(heights: np.ndarray) -> float:
    return float(np.abs(np.diff(heights)).sum())

def _height_variance(heights: np.ndarray) -> float:
    return float(heights.var())

def extract_features(board: np.ndarray, lines_cleared: int = 0) -> np.ndarray:
    """
    Compact feature vector fed to the value network.

    Layout (length = cols + 5):
        col heights (normalised)  |  agg_height  |  holes  |  bumpiness  |  variance  |  lines_cleared
    """
    rows, cols = board.shape
    h   = _col_heights(board)
    agg = h.sum()
    return np.array([
        *( h / rows ),
        agg         / (rows * cols),
        _count_holes(board) / (rows * cols),
        _bumpiness(h)       / (rows * cols),
        _height_variance(h) / (rows ** 2),
        float(lines_cleared),
    ], dtype=np.float32)


In [ ]:
#  Board simulator
def simulate_placement(board, piece_coords, col, rotation_angle, rows, cols):
    """
    Drop piece at (col, rotation) on a copy of board.
    Returns (new_board, lines_cleared) or None if invalid.
    """
    piece = piece_coords[rotation_angle]
    max_x = max(x for x, y in piece)
    min_x = min(x for x, y in piece)

    if col + min_x < 0 or col + max_x >= cols:
        return None

    # Hard drop
    drop_y = 0
    while True:
        colliding = False
        for x, y in piece:
            nx, ny = x + col, y + drop_y + 1
            if ny >= rows or (0 <= nx < cols and board[ny, nx] == 1):
                colliding = True; break
        if colliding: break
        drop_y += 1

    # Spawn collision check
    for x, y in piece:
        nx, ny = x + col, y + drop_y
        if not (0 <= ny < rows and 0 <= nx < cols): return None
        if board[ny, nx] == 1: return None

    new_board = board.copy()
    for x, y in piece:
        new_board[y + drop_y, x + col] = 1

    full = [r for r in range(rows) if new_board[r].sum() == cols]
    lines = len(full)
    if lines:
        new_board = np.delete(new_board, full, axis=0)
        new_board = np.concatenate([np.zeros((lines, cols), dtype=new_board.dtype), new_board])

    return new_board, lines

In [ ]:
# State/ Action design
class TetrisEnv:
    """
    get_next_states() → dict  { (col, rot_idx): (features, lines) }
    step(action_tuple) → (features, reward, done)

    Reward:
        w_lines 
    """

    w_lines    = 10.0
    w_survival =  0.1
    w_holes    =  0.8
    w_height   =  0.15
    w_bump     =  0.3
    w_var      =  0.5
    w_death    =  5.0

    def __init__(self, rows=14, cols=8, render=False):
        self._rows = rows; self._cols = cols
        self.feature_size     = cols + 5
        self._curriculum_level = 0
        self._game = Tetris(_rows=rows, _cols=cols, _render=render)
        self._done = False

    def reset(self):
        self._game.reset()
        if self._curriculum_level > 0:
            self._game.prefill_board(self._curriculum_level)
        self._done = False
        return extract_features(self._game.board)

    def get_next_states(self) -> dict:
        """Return features for every valid placement of the current piece."""
        results = {}
        piece_data = Tetris.tetris_pieces[self._game.current_piece]
        board = self._game.board
        seen = set()

        for rot_idx, angle in enumerate([0, 90, 180, 270]):
            for col in range(self._cols):
                out = simulate_placement(board, piece_data, col, angle,
                                         self._rows, self._cols)
                if out is None: continue
                new_board, lines = out
                key = new_board.tobytes()
                if key in seen: continue
                seen.add(key)
                results[(col, rot_idx)] = (extract_features(new_board, lines), lines)

        return results

    def step(self, action: tuple):
        """action = (col, rotation_index)"""
        col, rot_idx = action
        score_before = self._game.score

        self._game.step_col(col, rot_idx)

        lines = max(0, self._game.score - score_before)
        game_over = not self._game.is_alive() or self._game.pieces_placed > 10_000

        board = self._game.board
        h    = _col_heights(board)
        reward = (self.w_lines    * (lines ** 2))
        if game_over:
            reward -= self.w_death
            self._done = True

        feats = extract_features(board, lines_cleared=lines)
        return feats, float(reward), game_over

    def reduce_curriculum(self):
        if self._curriculum_level > 0:
            self._curriculum_level -= 1


In [ ]:
#  Value Network   V(s') → scalar
def build_value_network(feature_size: int, hidden=(128, 64, 32)) -> keras.Model:
    """Small MLP — input is the compact feature vector, output is one scalar."""
    inp = keras.Input(shape=(feature_size,), name="features")
    x   = inp
    for i, units in enumerate(hidden):
        x = keras.layers.Dense(units, name=f"fc{i}")(x)
        x = keras.layers.LayerNormalization()(x)
        x = keras.layers.ReLU()(x)
    out = keras.layers.Dense(1, name="value")(x)
    return keras.Model(inputs=inp, outputs=out, name="ValueNet")


In [ ]:
#  Replay Buffer

class ReplayBuffer:
    """Stores (state_features, reward, next_best_features, done)."""

    def __init__(self, capacity=50_000):
        self._buf = deque(maxlen=capacity)

    def push(self, state, reward, next_state, done):
        self._buf.append((state, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self._buf, batch_size)
        s, r, ns, d = zip(*batch)
        return (
            tf.constant(np.array(s,  dtype=np.float32)),
            tf.constant(np.array(r,  dtype=np.float32)),
            tf.constant(np.array(ns, dtype=np.float32)),
            tf.constant(np.array(d,  dtype=np.float32)),
        )

    def __len__(self): return len(self._buf)


In [ ]:
# ─────────────────────────────────────────────
#  Agent
# ─────────────────────────────────────────────

class TetrisAgent:
    """
    Afterstate value-network agent.
    act(next_states)        — epsilon-greedy
    act_greedy(next_states) — pure greedy (eval)
    """

    def __init__(
        self,
        feature_size,
        gamma         = 0.99,
        lr            = 1e-3,
        epsilon_start = 1.0,
        epsilon_end   = 0.01,
        epsilon_decay = 1_000,    # episodes to reach epsilon_end
        batch_size    = 512,
        buffer_size   = 50_000,
        tau           = 0.01,
        hidden        = (128, 64, 32),
    ):
        self.feature_size  = feature_size
        self.gamma         = gamma
        self.epsilon       = epsilon_start
        self.epsilon_start = epsilon_start
        self.epsilon_end   = epsilon_end
        self.epsilon_decay = epsilon_decay
        self.batch_size    = batch_size
        self.tau           = tau

        self.online_net = build_value_network(feature_size, hidden)
        self.target_net = build_value_network(feature_size, hidden)
        self._hard_update()

        self.optimizer  = keras.optimizers.Adam(learning_rate=lr, clipnorm=1.0)
        self.loss_fn    = keras.losses.Huber()
        self.buffer     = ReplayBuffer(capacity=buffer_size)
        self.train_steps = 0

    # ── Target-network sync ───────────────────────────────────────

    def _hard_update(self):
        self.target_net.set_weights(self.online_net.get_weights())

    def _soft_update(self):
        for ow, tw in zip(self.online_net.weights, self.target_net.weights):
            tw.assign(self.tau * ow + (1 - self.tau) * tw)

    # ── Epsilon schedule ──────────────────────────────────────────

    def update_epsilon(self, episode: int):
        """Linear decay from epsilon_start to epsilon_end over epsilon_decay episodes."""
        frac = min(1.0, episode / self.epsilon_decay)
        self.epsilon = self.epsilon_start + frac * (self.epsilon_end - self.epsilon_start)

    # ── Action selection ──────────────────────────────────────────

    def _score_states(self, next_states: dict, net) -> tuple:
        actions  = list(next_states.keys())
        features = np.array([next_states[a][0] for a in actions], dtype=np.float32)
        values   = net(features, training=False).numpy().flatten()
        return actions, values

    def act(self, next_states: dict) -> tuple:
        if random.random() < self.epsilon:
            return random.choice(list(next_states.keys()))
        actions, values = self._score_states(next_states, self.online_net)
        return actions[int(np.argmax(values))]

    def act_greedy(self, next_states: dict) -> tuple:
        actions, values = self._score_states(next_states, self.online_net)
        return actions[int(np.argmax(values))]

    # ── Learning ─────────────────────────────────────────────────

    def remember(self, state, reward, next_state, done):
        self.buffer.push(state, reward, next_state, done)

    @tf.function
    def _train_step(self, states, rewards, next_states, dones):
        next_values = tf.squeeze(self.target_net(next_states, training=False), axis=1)
        targets     = rewards + (1.0 - dones) * self.gamma * next_values

        with tf.GradientTape() as tape:
            predicted = tf.squeeze(self.online_net(states, training=True), axis=1)
            loss      = self.loss_fn(targets, predicted)

        grads = tape.gradient(loss, self.online_net.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.online_net.trainable_variables))
        return loss

    def learn(self):
        if len(self.buffer) < self.batch_size:
            return None
        s, r, ns, d = self.buffer.sample(self.batch_size)
        loss = self._train_step(s, r, ns, d)
        self._soft_update()
        self.train_steps += 1
        return float(loss)

    # ── Checkpointing ─────────────────────────────────────────────

    def make_checkpoint(self, ckpt_dir: str):
        ckpt = tf.train.Checkpoint(
            online_net  = self.online_net,
            target_net  = self.target_net,
            optimizer   = self.optimizer,
            train_steps = tf.Variable(self.train_steps, dtype=tf.int64),
            epsilon     = tf.Variable(self.epsilon,     dtype=tf.float32),
        )
        manager = tf.train.CheckpointManager(ckpt, ckpt_dir, max_to_keep=3)
        return ckpt, manager

    def restore_checkpoint(self, ckpt, manager):
        status = ckpt.restore(manager.latest_checkpoint)
        if manager.latest_checkpoint:
            self.train_steps = int(ckpt.train_steps.numpy())
            self.epsilon     = float(ckpt.epsilon.numpy())
            print(f"Restored checkpoint (train_steps={self.train_steps}, ε={self.epsilon:.4f})")
            return True
        print("No checkpoint found — starting fresh.")
        return False

    def save_weights(self, path):
        self.online_net.save_weights(path)
        print(f"Weights saved → {path}")

    def load_weights(self, path):
        self.online_net.load_weights(path)
        self._hard_update()
        print(f"Weights loaded ← {path}")


In [ ]:
# ─────────────────────────────────────────────
#  Config  (mirrors old CONFIG dict structure)
# ─────────────────────────────────────────────

CONFIG = {
    # Environment
    "rows":             14,
    "cols":             8,

    # Training schedule
    "episodes":         100_000,
    "warmup_episodes":  100,       # pure-random before learning

    # Agent / optimiser
    "lr":               1e-3,
    "gamma":            0.99,
    "epsilon_start":    1.0,
    "epsilon_end":      0.05,
    "epsilon_decay":    10_000,     # episodes for linear decay
    "batch_size":       512,
    "buffer_size":      50_000,
    "tau":              0.01,
    "hidden":           (128, 64, 32),

    # Curriculum
    "curriculum_start":     4,
    "curriculum_threshold": 50,    # avg pieces before reducing level
    "curriculum_window":    100,

    # Logging / saving
    "log_interval":     10,        # episodes between console prints
    "eval_interval":    500,       # episodes between eval runs
    "eval_episodes":    5,
    "save_interval":    250,       # episodes between checkpoints
    "checkpoint_dir":   "checkpoints",
    "policy_dir":       "saved_policy",
    "log_dir":          "logs",
    "log_name":         "training_log.csv",
    "tb_dir":           "logs/tb",
}

class _Cfg:
    def __init__(self, d): self.__dict__.update(d)


In [ ]:
# ─────────────────────────────────────────────
#  Evaluation helper
# ─────────────────────────────────────────────

def run_eval(agent: TetrisAgent, cfg: _Cfg, num_episodes: int, render: bool = False):
    rewards, pieces, scores = [], [], []
    for _ in range(num_episodes):
        env = TetrisEnv(rows=cfg.rows, cols=cfg.cols, render=render)
        env.reset()
        ep_reward = 0.0
        while True:
            ns = env.get_next_states()
            if not ns: break
            action = agent.act_greedy(ns)
            _, reward, done = env.step(action)
            ep_reward += reward
            if done: break
        rewards.append(ep_reward)
        pieces.append(env._game.get_pieces_placed())
        scores.append(env._game.score)
    return float(np.mean(rewards)), float(np.mean(pieces)), float(np.mean(scores))


In [ ]:
# ─────────────────────────────────────────────
#  Training loop
# ─────────────────────────────────────────────

def train(config: dict = None):
    cfg = _Cfg(config if config is not None else CONFIG)

    print("\n=== Tetris Afterstate Agent ===")
    print(f"Started: {datetime.datetime.now():%Y-%m-%d %H:%M:%S}\n")

    # ── Directories ───────────────────────────────────────────────
    ckpt_dir = Path(cfg.checkpoint_dir); ckpt_dir.mkdir(parents=True, exist_ok=True)
    policy_dir = Path(cfg.policy_dir);   policy_dir.mkdir(parents=True, exist_ok=True)
    log_dir  = Path(cfg.log_dir);        log_dir.mkdir(parents=True, exist_ok=True)
    tb_dir   = Path(cfg.tb_dir);         tb_dir.mkdir(parents=True, exist_ok=True)

    print(f"Checkpoint dir : {ckpt_dir.resolve()}")
    print(f"Policy dir     : {policy_dir.resolve()}")

    # ── TensorBoard ───────────────────────────────────────────────
    summary_writer = tf.summary.create_file_writer(str(tb_dir))
    print(f"TensorBoard    : {tb_dir.resolve()}")

    # ── CSV log ───────────────────────────────────────────────────
    csv_path   = log_dir / cfg.log_name
    csv_exists = csv_path.exists()
    csv_file   = open(csv_path, "a", newline="")
    fieldnames = ["episode", "timestamp", "total_reward", "pieces_placed",
                  "lines_cleared", "avg_reward_100", "avg_pieces_100",
                  "epsilon", "avg_loss_200", "eval_reward", "eval_pieces",
                  "eval_score", "curriculum_level"]
    csv_writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
    if not csv_exists:
        csv_writer.writeheader()
    print(f"CSV log        : {csv_path.resolve()}\n")

    # ── Environment + Agent ───────────────────────────────────────
    env = TetrisEnv(rows=cfg.rows, cols=cfg.cols)
    env._curriculum_level = cfg.curriculum_start

    agent = TetrisAgent(
        feature_size  = env.feature_size,
        gamma         = cfg.gamma,
        lr            = cfg.lr,
        epsilon_start = cfg.epsilon_start,
        epsilon_end   = cfg.epsilon_end,
        epsilon_decay = cfg.epsilon_decay,
        batch_size    = cfg.batch_size,
        buffer_size   = cfg.buffer_size,
        tau           = cfg.tau,
        hidden        = cfg.hidden,
    )

    # ── Checkpoint restore ────────────────────────────────────────
    ckpt, manager = agent.make_checkpoint(str(ckpt_dir))
    resumed = agent.restore_checkpoint(ckpt, manager)
    start_ep = agent.train_steps if resumed else 0

    # ── Rolling metrics ───────────────────────────────────────────
    rewards_win  = deque(maxlen=100)
    pieces_win   = deque(maxlen=cfg.curriculum_window)
    losses_win   = deque(maxlen=200)
    best_eval    = -float("inf")
    last_eval    = (0.0, 0.0, 0.0)

    # ── Main loop ─────────────────────────────────────────────────
    for episode in range(1, cfg.episodes + 1):
        env.reset()
        agent.update_epsilon(episode)

        total_reward  = 0.0
        pieces_placed = 0

        while True:
            next_states = env.get_next_states()
            if not next_states: break

            # Choose action
            if episode <= cfg.warmup_episodes:
                action = random.choice(list(next_states.keys()))
            else:
                action = agent.act(next_states)

            chosen_feats, lines = next_states[action]
            _, reward, done = env.step(action)
            total_reward  += reward
            pieces_placed  = env._game.pieces_placed

            # Build next-best features for TD target
            if done:
                next_best = np.zeros(env.feature_size, dtype=np.float32)
            else:
                nns = env.get_next_states()
                if nns:
                    na = list(nns.keys())
                    nf = np.array([nns[a][0] for a in na], dtype=np.float32)
                    nv = agent.online_net(nf, training=False).numpy().flatten()
                    next_best = nf[int(np.argmax(nv))]
                else:
                    next_best = np.zeros(env.feature_size, dtype=np.float32)

            agent.remember(chosen_feats, reward, next_best, float(done))

            if episode > cfg.warmup_episodes:
                loss = agent.learn()
                if loss is not None:
                    losses_win.append(loss)

            if done: break

        rewards_win.append(total_reward)
        pieces_win.append(pieces_placed)

        avg_reward = float(np.mean(rewards_win))
        avg_pieces = float(np.mean(pieces_win))
        avg_loss   = float(np.mean(losses_win)) if losses_win else float("nan")

        # ── Curriculum ────────────────────────────────────────────
        if (len(pieces_win) == cfg.curriculum_window
                and avg_pieces > cfg.curriculum_threshold
                and env._curriculum_level > 0):
            env.reduce_curriculum()
            print(f"[Curriculum ↓] Level → {env._curriculum_level} (avg pieces={avg_pieces:.1f})")
            pieces_win.clear()

        # ── Eval ──────────────────────────────────────────────────
        eval_reward = eval_pieces = eval_score = float("nan")
        if episode % cfg.eval_interval == 0:
            eval_reward, eval_pieces, eval_score = run_eval(agent, cfg, cfg.eval_episodes)
            last_eval = (eval_reward, eval_pieces, eval_score)
            print(f"[{episode:>6}]  *** eval reward={eval_reward:.2f} | "                  f"pieces={eval_pieces:.1f} | score={eval_score:.2f} ***")
            with summary_writer.as_default():
                tf.summary.scalar("eval/mean_reward",   eval_reward, step=episode)
                tf.summary.scalar("eval/avg_pieces",    eval_pieces, step=episode)
                tf.summary.scalar("eval/avg_score",     eval_score,  step=episode)

        # ── TensorBoard scalars ───────────────────────────────────
        with summary_writer.as_default():
            tf.summary.scalar("train/reward",   total_reward, step=episode)
            tf.summary.scalar("train/pieces",   pieces_placed, step=episode)
            tf.summary.scalar("train/epsilon",  agent.epsilon, step=episode)
            if not np.isnan(avg_loss):
                tf.summary.scalar("train/loss", avg_loss, step=episode)

        # ── CSV row ───────────────────────────────────────────────
        csv_writer.writerow({
            "episode":         episode,
            "timestamp":       datetime.datetime.now().isoformat(),
            "total_reward":    round(total_reward, 3),
            "pieces_placed":   pieces_placed,
            "lines_cleared":   env._game.score,
            "avg_reward_100":  round(avg_reward, 3),
            "avg_pieces_100":  round(avg_pieces, 2),
            "epsilon":         round(agent.epsilon, 5),
            "avg_loss_200":    round(avg_loss, 5) if not np.isnan(avg_loss) else "",
            "eval_reward":     round(eval_reward, 3) if not np.isnan(eval_reward) else "",
            "eval_pieces":     round(eval_pieces, 2) if not np.isnan(eval_pieces) else "",
            "eval_score":      round(eval_score, 2) if not np.isnan(eval_score) else "",
            "curriculum_level": env._curriculum_level,
        })
        csv_file.flush()

        # ── Console log ───────────────────────────────────────────
        if episode % cfg.log_interval == 0:
            print(
                f"[{episode:>6}]  "                f"R {total_reward:>7.1f}  AvgR {avg_reward:>7.1f}  "                f"Pcs {pieces_placed:>4}  AvgPcs {avg_pieces:>5.1f}  "                f"Lines {env._game.score:>3}  "                f"ε {agent.epsilon:.4f}  Loss {avg_loss:.4f}"
            )

        # ── Checkpoint + policy save ──────────────────────────────
        if episode % cfg.save_interval == 0 and episode > 0:
            ckpt.train_steps.assign(agent.train_steps)
            ckpt.epsilon.assign(agent.epsilon)
            manager.save()
            agent.online_net.export(str(policy_dir / f"ep_{episode}"))
            print(f"[{episode:>6}]  Saved checkpoint + policy.")
            if avg_reward > best_eval:
                best_eval = avg_reward
                agent.save_weights(str(policy_dir / "best.weights.h5"))

    csv_file.close()
    er, ep2, es = run_eval(agent, cfg, cfg.eval_episodes * 2)
    print(f"\nTraining complete — reward={er:.2f} | pieces={ep2:.1f} | score={es:.2f}")
    print(f"Policy saved to : {policy_dir.resolve()}")

    return agent


In [ ]:
agent = train()


In [ ]:
# Resume from checkpoint — just call train() again; checkpoint restores automatically
agent = train()


In [ ]:
pip install matplotlib

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(f"logs/{CONFIG['log_name']}")

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle("Tetris Afterstate Agent — Training Metrics", fontsize=14)

# Loss
ax = axes[0, 0]
loss_vals = pd.to_numeric(df["avg_loss_200"], errors="coerce").dropna()
ax.plot(df.loc[loss_vals.index, "episode"], loss_vals)
ax.set_title("Training Loss (avg 200)"); ax.set_xlabel("Episode"); ax.set_ylabel("Huber Loss")

# Avg reward
ax = axes[0, 1]
ax.plot(df["episode"], df["avg_reward_100"], color="green")
ax.set_title("Avg Reward (100 ep)"); ax.set_xlabel("Episode"); ax.set_ylabel("Reward")

# Pieces placed
ax = axes[1, 0]
ax.plot(df["episode"], df["avg_pieces_100"], color="purple")
ax.set_title("Avg Pieces Placed (100 ep)"); ax.set_xlabel("Episode"); ax.set_ylabel("Pieces")

# Epsilon
ax = axes[1, 1]
ax.plot(df["episode"], df["epsilon"], color="orange")
ax.set_title("Epsilon (Exploration)"); ax.set_xlabel("Episode"); ax.set_ylabel("ε")

plt.tight_layout()
plt.savefig("training_metrics.png", dpi=150)
plt.show()

# Eval metrics (if any evals ran)
eval_df = df[df["eval_reward"] != ""].copy()
if len(eval_df) > 0:
    eval_df["eval_reward"] = pd.to_numeric(eval_df["eval_reward"])
    eval_df["eval_pieces"] = pd.to_numeric(eval_df["eval_pieces"])
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(eval_df["episode"], eval_df["eval_reward"], color="green", marker="o")
    axes[0].set_title("Eval Reward"); axes[0].set_xlabel("Episode")
    axes[1].plot(eval_df["episode"], eval_df["eval_pieces"], color="blue", marker="o")
    axes[1].set_title("Eval Pieces Placed"); axes[1].set_xlabel("Episode")
    plt.tight_layout()
    plt.savefig("eval_metrics.png", dpi=150)
    plt.show()
